# Random Forest - interview style walkthrough


This notebook builds a Random Forest model, explains typical interview cases, and ends with a take-home style assignment.


In [1]:
# Reference
# https://medium.com/@abhishekjainindore24/everything-about-random-forest-90c106d63989


## 1) Build a model
We'll use a classic binary classification dataset so the focus stays on the Random Forest itself.


In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [2]:
# Load data
raw = load_breast_cancer()
X = pd.DataFrame(raw.data, columns=raw.feature_names)
y = pd.Series(raw.target, name="target")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [3]:
# Train model
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1,
)

rf.fit(X_train, y_train)


,n_estimators,300
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [4]:
# Evaluate
pred = rf.predict(X_test)
print("Accuracy:", accuracy_score(y_test, pred))
print("Confusion matrix:", confusion_matrix(y_test, pred))
print("Classification report:", classification_report(y_test, pred))


Accuracy: 0.9473684210526315
Confusion matrix: [[39  3]
 [ 3 69]]
Classification report:               precision    recall  f1-score   support

           0       0.93      0.93      0.93        42
           1       0.96      0.96      0.96        72

    accuracy                           0.95       114
   macro avg       0.94      0.94      0.94       114
weighted avg       0.95      0.95      0.95       114



In [5]:
# Feature importance (top 10)
importances = pd.Series(rf.feature_importances_, index=X.columns)
print(importances.sort_values(ascending=False).head(10))


worst perimeter         0.137344
worst area              0.137312
worst concave points    0.115502
mean concave points     0.091774
worst radius            0.084111
mean radius             0.062741
mean perimeter          0.049680
mean concavity          0.044142
mean area               0.040508
worst concavity         0.034837
dtype: float64


- **Imbalanced classes**: Use `class_weight='balanced'`, stratified splits, and metrics like ROC-AUC/PR-AUC.
- **High dimensionality**: Trees can handle it, but control depth, `max_features`, and use permutation importance.
- **Outliers and non-linear patterns**: Random Forests are robust here; compare to linear baselines.
- **Missing values**: Impute (median/most_frequent) or use models/implementations that handle missingness.
- **Overfitting**: Increase `min_samples_leaf`, restrict depth, or use more trees with OOB scoring.
- **Interpretability**: Use feature importances, partial dependence, or permutation importance.


### Task


You are given a dataset for binary classification. Your task is to build and justify a Random Forest solution.

Deliverables:
- A baseline model with clear train/test split and evaluation metrics.
- A tuned model using cross-validation or a validation set.
- A short explanation of hyperparameters you chose and why.
- A brief comparison against a simple baseline (e.g., Logistic Regression).
- A small write-up explaining how you would handle class imbalance and feature importance.


In [22]:
# Dummy dataset for the assignment
from sklearn.datasets import make_classification

X_dummy, y_dummy = make_classification(
    n_samples=800,
    n_features=12,
    n_informative=6,
    n_redundant=2,
    n_classes=2,
    weights=[0.7, 0.3],
    random_state=42,
)

# TODO: starter template for the assignment
# 1) Split X_dummy, y_dummy into train/validation/test
# 2) Train a baseline RandomForestClassifier
# 3) Tune hyperparameters (GridSearchCV or RandomizedSearchCV)
# 4) Compare with LogisticRegression
# 5) Summarize results and explain feature importances
# 6 ) Compare it with other boosting algorithms (XGBoost, LightGBM, CatBoost)


In [23]:
Xd_train, Xd_test, yd_train, yd_test = train_test_split(
    X_dummy, y_dummy, test_size=0.2, random_state=42, stratify=y_dummy
)

In [24]:
# Train model
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1,
    class_weight='balanced'
)

rf.fit(Xd_train, yd_train)

,n_estimators,300
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [31]:
# Evaluate
pred = rf.predict(Xd_test)
rf.feature_importances_

print("Accuracy:", accuracy_score(yd_test, pred))
print("Confusion matrix:", confusion_matrix(yd_test, pred))
print("Classification report:", classification_report(yd_test, pred))

Accuracy: 0.9
Confusion matrix: [[105   6]
 [ 10  39]]
Classification report:               precision    recall  f1-score   support

           0       0.91      0.95      0.93       111
           1       0.87      0.80      0.83        49

    accuracy                           0.90       160
   macro avg       0.89      0.87      0.88       160
weighted avg       0.90      0.90      0.90       160



In [33]:
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
    'max_features': ['sqrt', 'log2'],
    'class_weight': ['balanced']
}
from sklearn.model_selection import GridSearchCV

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    n_jobs=-1,
    verbose=1
)

grid_search.fit(Xd_train, yd_train)


Fitting 5 folds for each of 48 candidates, totalling 240 fits


,estimator,RandomForestC...ndom_state=42)
,param_grid,"{'class_weight': ['balanced'], 'max_depth': [None, 10, ...], 'max_features': ['sqrt', 'log2'], 'min_samples_leaf': [1, 2], ...}"
,scoring,None
,n_jobs,-1
,refit,True
,cv,5
,verbose,1
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,200


In [ ]:
best_params = grid_search.best_params_
print("Best hyperparameters:", best_params)

# you can re-train the model with the best hyperparameters
best_model = RandomForestClassifier(**best_params)
best_model.fit(Xd_train, yd_train)

# evaluate the model
y_pred_best = best_model.predict(Xd_test)
accuracy = accuracy_score(yd_test, y_pred_best)
print("Accuracy with best hyperparameters:", accuracy)

# pred = rf.predict(Xd_test)
# print("Accuracy:", accuracy_score(yd_test, y_pred_best))
print("Confusion matrix:", confusion_matrix(yd_test, y_pred_best))
print("Classification report:", classification_report(yd_test, y_pred_best))

Best hyperparameters: {'class_weight': 'balanced', 'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}
Accuracy with best hyperparameters: 0.9125
Confusion matrix: [[105   6]
 [  8  41]]
Classification report:               precision    recall  f1-score   support

           0       0.93      0.95      0.94       111
           1       0.87      0.84      0.85        49

    accuracy                           0.91       160
   macro avg       0.90      0.89      0.90       160
weighted avg       0.91      0.91      0.91       160



In [15]:
from sklearn.linear_model import LogisticRegression


model = LogisticRegression()
model.fit(Xd_train, yd_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [16]:
y_pred = model.predict(X_test)
accuracy_score(y_test, y_pred)

0.84375

In [ ]:
# hyper parameter tuning
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'solver': ['lbfgs', 'liblinear', 'newton-cg']
}

grid_search = GridSearchCV(LogisticRegression(), param_grid, cv=5,verbose=1)
grid_search.fit(X_train, y_train)

Fitting 5 folds for each of 18 candidates, totalling 90 fits
Best hyperparameters: {'C': 0.1, 'penalty': 'l2', 'solver': 'lbfgs'}


/Users/sathish/Documents/akila/ML-learning-materials/ai-ml-akila/ml-env/lib/python3.14/site-packages/sklearn/model_selection/_validation.py:516: FitFailedWarning: 
30 fits failed out of a total of 90.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
15 fits failed with the following error:
Traceback (most recent call last):
  File "/Users/sathish/Documents/akila/ML-learning-materials/ai-ml-akila/ml-env/lib/python3.14/site-packages/sklearn/model_selection/_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/sathish/Documents/akila/ML-learning-materials/ai-ml-akila/ml-env/lib/python3.14/site-packages/sklearn/base.py", line 1365, in wra

In [18]:
# best hyperparameters
best_params = grid_search.best_params_
print("Best hyperparameters:", best_params)

# you can re-train the model with the best hyperparameters
best_model = LogisticRegression(**best_params)
best_model.fit(X_train, y_train)

# evaluate the model
y_pred_best = best_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred_best)
print("Accuracy with best hyperparameters:", accuracy)

Best hyperparameters: {'C': 0.1, 'penalty': 'l2', 'solver': 'lbfgs'}
Accuracy with best hyperparameters: 0.84375


Logistic Regression provides a fast, interpretable baseline, while Random Forest improves performance by capturing non-linear patterns at the cost of interpretability and computational efficiency.